In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# SIGIZI - Random Forest Training for Stunting Classification\n",
    "\n",
    "Notebook ini melatih model **Random Forest** untuk mengklasifikasikan status stunting balita (Normal, Stunting, Severe Stunting) berdasarkan data antropometri.\n",
    "\n",
    "**Fitur input:**\n",
    "- umur_bulan (usia dalam bulan)\n",
    "- jenis_kelamin (L/P, akan di-encode)\n",
    "- berat_kg\n",
    "- tinggi_cm\n",
    "- zscore_bb_u (opsional, tapi membantu akurasi)\n",
    "- zscore_tb_u (opsional)\n",
    "\n",
    "**Target:** status_stunting (3 kelas)\n",
    "\n",
    "Keluaran:\n",
    "- Model `random_forest_stunting.pkl`\n",
    "- Label encoder untuk gender `gender_encoder.pkl` (optional)\n",
    "- Classification report, confusion matrix, feature importance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import joblib\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "\n",
    "print(\"Libraries loaded.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load dataset\n",
    "DATA_PATH = \"../stunting_dataset.csv\"   # karena notebook di folder ml-training/\n",
    "df = pd.read_csv(DATA_PATH)\n",
    "print(f\"Total samples: {len(df)}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Encode jenis_kelamin (L -> 1, P -> 0 atau sebaliknya)\n",
    "le_gender = LabelEncoder()\n",
    "df['gender_encoded'] = le_gender.fit_transform(df['jenis_kelamin'])\n",
    "print(\"Mapping gender:\", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))\n",
    "\n",
    "# Feature selection\n",
    "features = ['umur_bulan', 'gender_encoded', 'berat_kg', 'tinggi_cm', 'zscore_bb_u', 'zscore_tb_u']\n",
    "X = df[features]\n",
    "y = df['status_stunting']  # target sudah dalam bentuk kategorikal\n",
    "\n",
    "print(f\"Feature shape: {X.shape}\")\n",
    "print(f\"Target classes: {y.unique()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data (80% train, 20% test) dengan stratifikasi\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42, stratify=y\n",
    ")\n",
    "\n",
    "print(f\"Train size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)\")\n",
    "print(f\"Test size : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)\")\n",
    "print(\"\\nDistribusi kelas pada train:\")\n",
    "print(y_train.value_counts(normalize=True))\n",
    "print(\"\\nDistribusi kelas pada test:\")\n",
    "print(y_test.value_counts(normalize=True))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Training Model Random Forest\n",
    "Hyperparameter awal: `n_estimators=200, max_depth=10` (sesuai keinginan user).\n",
    "Kita akan lakukan juga pencarian parameter sederhana untuk meningkatkan akurasi."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Model awal\n",
    "model = RandomForestClassifier(\n",
    "    n_estimators=200,\n",
    "    max_depth=10,\n",
    "    random_state=42,\n",
    "    class_weight='balanced'  # karena data imbalance (normal > stunting)\n",
    ")\n",
    "\n",
    "model.fit(X_train, y_train)\n",
    "y_pred = model.predict(X_test)\n",
    "acc = accuracy_score(y_test, y_pred)\n",
    "print(f\"Accuracy awal: {acc:.4f}\")\n",
    "print(\"\\nClassification Report:\")\n",
    "print(classification_report(y_test, y_pred))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Confusion Matrix\n",
    "cm = confusion_matrix(y_test, y_pred, labels=['Normal', 'Stunting', 'Severe Stunting'])\n",
    "plt.figure(figsize=(6,5))\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Stunting', 'Severe Stunting'], yticklabels=['Normal', 'Stunting', 'Severe Stunting'])\n",
    "plt.title('Confusion Matrix')\n",
    "plt.xlabel('Predicted')\n",
    "plt.ylabel('Actual')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Feature Importance\n",
    "Mengetahui fitur mana yang paling berpengaruh."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "importances = model.feature_importances_\n",
    "feature_names = features\n",
    "indices = np.argsort(importances)[::-1]\n",
    "\n",
    "plt.figure(figsize=(8,4))\n",
    "plt.title('Feature Importance')\n",
    "plt.bar(range(len(importances)), importances[indices], align='center')\n",
    "plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "for i in indices:\n",
    "    print(f\"{feature_names[i]}: {importances[i]:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Hyperparameter Tuning (Opsional)\n",
    "Mencari parameter terbaik untuk meningkatkan akurasi di atas 95%."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Parameter grid sederhana\n",
    "param_grid = {\n",
    "    'n_estimators': [100, 200, 300],\n",
    "    'max_depth': [8, 10, 12, None],\n",
    "    'min_samples_split': [2, 5, 10]\n",
    "}\n",
    "\n",
    "# GridSearch dengan cross-validation 5 fold\n",
    "# (Proses ini memakan waktu, bisa dilewati jika ingin cepat)\n",
    "grid_search = GridSearchCV(RandomForestClassifier(random_state=42, class_weight='balanced'), \n",
    "                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)\n",
    "grid_search.fit(X_train, y_train)\n",
    "\n",
    "print(\"Best parameters:\", grid_search.best_params_)\n",
    "print(\"Best CV accuracy:\", grid_search.best_score_)\n",
    "\n",
    "# Evaluasi dengan best model\n",
    "best_model = grid_search.best_estimator_\n",
    "y_pred_best = best_model.predict(X_test)\n",
    "acc_best = accuracy_score(y_test, y_pred_best)\n",
    "print(f\"Test accuracy after tuning: {acc_best:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gunakan best model jika ada, jika tidak pakai model awal\n",
    "final_model = best_model if 'best_model' in locals() else model\n",
    "final_accuracy = accuracy_score(y_test, final_model.predict(X_test))\n",
    "print(f\"Final Model Accuracy: {final_accuracy:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Simpan Model dan Encoder\n",
    "Simpan ke folder `model/` (pastikan folder sudah ada)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.makedirs(\"../model\", exist_ok=True)\n",
    "\n",
    "joblib.dump(final_model, \"../model/random_forest_stunting.pkl\")\n",
    "joblib.dump(le_gender, \"../model/gender_encoder.pkl\")\n",
    "\n",
    "print(\"Model saved to ../model/random_forest_stunting.pkl\")\n",
    "print(\"Gender encoder saved to ../model/gender_encoder.pkl\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Contoh loading dan prediksi dengan model yang disimpan\n",
    "loaded_model = joblib.load(\"../model/random_forest_stunting.pkl\")\n",
    "sample = X_test.iloc[[0]]\n",
    "pred = loaded_model.predict(sample)\n",
    "print(f\"Contoh prediksi untuk sample pertama test set: {pred[0]}\")\n",
    "print(f\"Actual: {y_test.iloc[0]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Kesimpulan\n",
    "Model Random Forest berhasil dilatih dengan akurasi `final_accuracy`. Model ini akan diintegrasikan ke dalam backend Flask untuk klasifikasi Z-score secara otomatis."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

NameError: name 'null' is not defined